# Step 5a — MuseTalk real-time benchmark on a Kaggle T4

Question this answers: **can a free T4 lip-sync a face in real time?** MuseTalk 1.5 is run on its
own sample avatar and audio; we measure frames/second and GPU memory. Real-time needs **≥ 25 fps**
of generated frames (the avatar video's frame rate), with headroom.

Nothing here touches the agent worker. Settings: **GPU T4 x2**, **Internet On**. No secrets needed.

**Start from a fresh session** (Run → Factory reset, or stop and restart) if an earlier version of
this notebook ran: its OpenMMLab cell downgraded `requests`/`tqdm`/`urllib3`/`setuptools`.

MuseTalk's avatar preparation normally needs OpenMMLab (`mmpose`), which can't be installed on
Python 3.13. Instead, this notebook drops in `kaggle/musetalk_patches/preprocessing.py` from our
repo: same face box (MuseTalk's own S3FD) and the same 68-point landmarks, from `face-alignment`
(pure PyTorch). It runs once per avatar, not per frame, so it doesn't affect the speed measured.

In [ ]:
import sys, subprocess
print(sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch; print("torch", torch.__version__, "cuda", torch.version.cuda, torch.cuda.is_available())

In [ ]:
MT = "/kaggle/working/MuseTalk"
!rm -rf {MT} && git clone -q --depth 1 https://github.com/TMElyralab/MuseTalk.git {MT}
!cd {MT} && git log --oneline -1 && ls

In [ ]:
# MuseTalk's requirements.txt pins versions that don't exist for Python 3.13 (numpy 1.23,
# tensorflow 2.12); install the libraries it actually imports, unpinned, on top of Kaggle's torch.
!pip install -q diffusers accelerate transformers einops omegaconf librosa soundfile \
    opencv-python-headless "imageio[ffmpeg]" ffmpeg-python huggingface_hub gdown

In [ ]:
# Replace MuseTalk's mmpose-based landmark step with our face-alignment version.
!pip install -q face-alignment
import urllib.request
PATCH = ("https://raw.githubusercontent.com/daneshrooman-del/Rooman-Agent/"
         "feature/realtime-voice-loop/kaggle/musetalk_patches/preprocessing.py")
urllib.request.urlretrieve(PATCH, f"{MT}/musetalk/utils/preprocessing.py")
!grep -n "mmpose\|face_alignment" {MT}/musetalk/utils/preprocessing.py | head -5

In [ ]:
# Weights, laid out the way MuseTalk's download_weights.sh expects (models/...).
import os, urllib.request
from huggingface_hub import hf_hub_download
M = f"{MT}/models"
def hf(repo, filename, dest_dir, **kw):
    os.makedirs(dest_dir, exist_ok=True)
    return hf_hub_download(repo, filename, local_dir=dest_dir, **kw)

for f in ["musetalkV15/unet.pth", "musetalkV15/musetalk.json"]:
    hf("TMElyralab/MuseTalk", f, M)
for f in ["config.json", "diffusion_pytorch_model.bin"]:
    hf("stabilityai/sd-vae-ft-mse", f, f"{M}/sd-vae")
for f in ["config.json", "pytorch_model.bin", "preprocessor_config.json"]:
    hf("openai/whisper-tiny", f, f"{M}/whisper")
os.makedirs(f"{M}/face-parse-bisent", exist_ok=True)
!gdown -q --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O {M}/face-parse-bisent/79999_iter.pth
urllib.request.urlretrieve("https://download.pytorch.org/models/resnet18-5c106cde.pth",
                           f"{M}/face-parse-bisent/resnet18-5c106cde.pth")
!find {M} -type f | sed "s|{M}/||" | sort

In [ ]:
# Benchmark: MuseTalk's own real-time script on its sample avatar + sample audio.
# --skip_save_images makes it report pure generation speed. GPU memory is sampled alongside.
!cat {MT}/configs/inference/realtime.yaml
mem_log = "/kaggle/working/gpu_mem.csv"
sampler = subprocess.Popen(f"nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 1 -i 0 > {mem_log}", shell=True)
cmd = (f"cd {MT} && python -m scripts.realtime_inference --inference_config configs/inference/realtime.yaml "
       "--result_dir results/realtime --unet_model_path models/musetalkV15/unet.pth "
       "--unet_config models/musetalkV15/musetalk.json --version v15 --fps 25 "
       "--batch_size 20 --skip_save_images")
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
sampler.terminate()
print("exit", r.returncode)
print("\n".join(l for l in (r.stdout + r.stderr).splitlines()
                if any(k in l.lower() for k in ("process time", "fps", "frames", "error", "traceback")))[-3000:])
mem = [int(x) for x in open(mem_log).read().split() if x.isdigit()]
print(f"peak GPU memory: {max(mem) if mem else 'n/a'} MiB")

**Reading the result:** the script prints `Total process time of N frames ... = T s`.
Throughput = N / T frames per second. ≥ 25 fps → real time works on a T4; 15–25 → workable with a
lower avatar fps or smaller batches; < 15 → use a lighter renderer for the live path.

The first run also prepares the avatar (face boxes, latents, masks) before generating — that part
is one-time per avatar and is not the number that matters.

If the run failed, paste the output. Most likely: a newer `diffusers`/`transformers` API than
MuseTalk was written for.